# Web Scraping + File I/O

##### Today's Topics:
1. Urllib and Beautiful Soup
2. Selenium
3. File Input/Output

* This is likely the most important day in the course (along with day05 on APIs).  
* You will use all the modules here if you want to scrape the internet.

***

### 1. Web Scraping (without APIs)

* Web scraping is the art of extracting data from websites and delivering it in formats like JSON, CSV, HTML, PDF, etc.
* Web scraping can be done either by using coding languages like Python, or by using data extraction APIs (Day 5).

##### Benefits 

1. Time-saving
2. Data accuracy
3. Cost-effective 

##### Ethics 

- Use a Public API when available and avoid scraping all together if the data you are looking if available through the API
- Only scrape when it is legal! 
    - NOT all sites can be legally scraped. Please don't get sued. 
    - Always check terms of service.
    - When in doubt, ask or don't do it. 
- Be polite and don't break websites
    - Scrape your data at a reasonable rate and control the number of requests per second. 
    - You don't want the website owner to think it as a DDoS attack. 

##### Overview of Web Scraping (without APIs)

1. Call the website and open it
2. Extract or load all the html code (you can store it locally for later use)
3. Retrieve information using the names of the tags, ids, etc. 
4. Store the data in to files (like csv)

#### 1.1 The Skeleton HTML Layout

In [ ]:
# <!DOCTYPE html> <html>
# <head>
# <title> Page Title </title>
# </head>
# <body>

# <h1>My first heading </h1>
# <p>My first paragraph. </p>

# </body> 
# </html>

_See https://www.w3schools.com/tags/default.asp for a list of HTML tags_

##### Let's look at some source code!

* Now go to https://polisci.wustl.edu/people/88/ 
* Click right, then View Page Source or (more likely) Inspect

##### 

#### 1.2 Web Crawlers

##### We mainly use two libraries: `urllib` and `BeautifulSoup`

1. `urllib`:
    - web crawler 
    - navigates to a url
2. `BeautifulSoup`
    - parses a downloaded HTML


Useful when:
- Info is contained in HTML (not served by JavaScript)
- Encoded HTML follows predictable pattern
- Example: https://www.presidency.ucsb.edu/documents/app-categories/presidential

Beautiful Soup documentation: 
http://www.crummy.com/software/BeautifulSoup/bs4/doc/

* Run the line below in a Jupyter cell if not installed alreay

In [ ]:
# ! pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import ssl
import certifi

#### 1.3 Example (WUSTL Political Science Webpage):

1. Open a web page

In [ ]:
web_address = 'https://polisci.washu.edu/people'
web_page = urllib.request.urlopen(web_address)
web_page #stored on machine

In [ ]:
#Try these alternative lines if the above didn't run

# import requests

# response = requests.get(web_address, verify=False)
# web_page = response.text

2. Parse it

In [ ]:
soup = BeautifulSoup(web_page.read())
print(soup)
print(soup.prettify()) # enable us to view how tags are nested in the document

In [ ]:
str(soup.prettify()) # enable us to view how tags are nested in the document

In [ ]:
str(soup.prettify())[0:1500]

3. Find all cases of a certain tag 'a'

In [ ]:
soup.find_all('a')[2:10] # Returns a list... remember this! # a is for anchor tag

4. Find all cases of a certain tag `<h3>`

In [ ]:
soup.find_all('h3')[2:12]

5. Extract text from the tag

In [ ]:
names = soup.find_all('h3') # list of html entries
[i.text for i in names][2:10] # grab just the text from each one

* We can create an object containing all elements with the tag `<a>`. Then, get the attributes

In [ ]:
all_a_tags = soup.find_all('a')
# all_a_tags
all_a_tags[36].attrs  # returns a dictionary with the attributes

* Access the attributes with key-value syntax

In [ ]:
all_a_tags[36].attrs.keys()

In [ ]:
all_a_tags[36]['href']

In [ ]:
for i in range(34,40):
  print(all_a_tags[i]['href'])

##### Some notes

*  Careful for the first and last tags—these can often be different than the others

In [ ]:
all_a_tags[0].attrs

* Because `all_a_tags` is a list, we need to index the element(s) we're interested in
* If we are interested in the first instance of the tag `<a>`, we can use

In [ ]:
soup.find('a')

In [ ]:
soup.find('a').attrs # meta data instead of the whole chunk

We can use a loop (for or while) to get and re-organize all the data.

In [ ]:
l = {"class" : [], "href" : []} # create a dictionary
for p in range(20,43):
    l["href"].append(all_a_tags[p].attrs["href"]) 

print(l)

##### If we are interested only in the attributes of `class = card` nested within tag 'a', we can specify this in our initial `find_all()` call:

In [ ]:
soup.find_all('a', {'class' : "card"})[0:2] # returns a list

##### Commonly, you will need to go level by level in an exporatory exercise to access nested tags. Here is an example:

* First get all tags `<div>`

In [ ]:
sections = soup.find_all('div') 
len(sections) # check the size of the object

* View the FIRST `<a>` tag within the first valid `<div>` tag 

In [ ]:
sections[2].a

* Or, equivalently:

In [ ]:
sections[2].find('a') 

* This gives us ALL `<a>` tags within the first valid `<div>` tag 

In [ ]:
sections[2].find_all('a')[0:10] 

* This gives us ALL `<a>` tags within the first valid `<div>` tag where `class` is 'first-level'

In [ ]:
sections[2].find_all('a', {'class' : 'logo'}) 

##### We can also create a tree of objects. Here is an example: 

Let's find Prof. Taylor Carlson's profile on the department website. 
1. Find all `<a>` tags where `class` is 'card'

In [ ]:
taylor = soup.find("a", {"aria-label": lambda x: x and "Taylor Carlson" in x}) # x exists and contains "Taylor Carlson"
print(taylor)


2. Manually examine where Prof. Carlson is located at. 

3. Find the heading that contains Prof. Carlson's first and last name.

In [ ]:
taylor.find_all('h3')
# taylor.find('h3').text

4. Check the contents contained within this `<a>` tag for Prof. Carlson. 
Notice that this is basically the same output as above, but without the `<a></a>` tags. So it is returning everything nested within the 'a' tag.

In [ ]:
taylor.contents

* This is an iterator
* Remember: iterators are objects that we access with loops

In [ ]:
taylor.children # generator object, memory efficient, faster setup, and infinite sequence 
# produces values one at a time instead of building the whole list in memory

# from bs4 import Tag

# only_tags = [c for c in taylor.children if isinstance(c, Tag)]
# only_tags[0].text # get the text from the first tag <tag>Taylor</tag>

5. Print all nested elements within 'taylor'

In [ ]:
import time
import random
import os

def download_page(address, filename, wait = 5):
  time.sleep(random.uniform(0,wait))
  
  page = urllib.request.urlopen(address)
  page_content = page.read()
  if os.path.exists(filename) == False:
    with open(filename, 'w') as p_html:
      p_html.write(str(page_content)) # needed to cast as string
  else:
    print("Can't overwrite file " + filename)

download_page('https://polisci.wustl.edu/people', "polisci_ppl.html")

In [ ]:
# list(card.children)     # [<h3>...</h3>, <p>...</p>]
# card.contents           # [<h3>...</h3>, <p>...</p>]
# list(card.descendants)  # [<h3>, "Taylor", <p>, "Professor of Political Science"]

In [ ]:
# there is only one child element in this case
for i, child in enumerate(taylor.children):
    print("Child %d: %s" % (i,child), '\n') 

##### Let's now look at sibling tags of `taylor`

In [ ]:
# Siblings (Example):

# <html>
#   <body>
#       <a>
#         <b>
#          text1
#         </b>
#         <c>
#          text2
#         </c>
#       </a>
#   </body>
# </html>


# Which two tags are on the same level? 

* See siblings *after* `taylor` in the sequence of `<a>` tags

In [ ]:
from bs4 import NavigableString


siblings = [s for s in taylor.next_siblings if not isinstance(s, NavigableString)]
for sib in siblings[:2]:
    print(sib.name, sib.get("aria-label"), sib.get("href"))

* Or see siblings *before* `taylor` in the sequence of `<a>` tags

#### 1.4 Crawler Detection

##### Crawlers are incredibly fast, but also easier to detect and block. 

You can incorporate some pauses to avoid detection. Strategies include: 

1. Using a random number generator to sleep for a random number of seconds
2. After each iteration, sleep for a fixed number of seconds

##### Import module `random` to generate random numbers, and module `time` to control the pauses in your code

* Random-second pause approach

In [ ]:
import random
import time

# Script will pause for n seconds
time.sleep(random.uniform(1, 5))
print('Pause Ended')

* Fixed-second pause approach

In [ ]:
time.sleep(5)
print('done')

#### 1.5 Remote Drivers

##### Selenium is a “remote driver” of your favorite browser. 

* You can pretty much simulate behavior of a human “surfing the web”. 
* With the right tricks, the likelihood of tracking and blocking your “bot” decreases.
* It also offers flexibility in terms of “unknown” items: you can even look by name of buttons in the page. 

##### There are some downsides though...
  - It is slower
  - It is dependent on your internet connection quality

##### Let's walk through an example using Selenium

* If you haven't already, make sure to install `Selenium` by running
    * `pip install selenium` in terminal or command line, or
    * `!pip install selenium` in a Jupyter notebook cell

download appropriate web driver from browser, e.g. https://chromedriver.chromium.org/downloads


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys

1. Give the path to your driver.

In [ ]:
import os

In [ ]:
os.getcwd()

In [ ]:
# # Interactive example:

from selenium import webdriver
driver = webdriver.Chrome()
#If you dont have the chrome driver, uncomment the following lines, comment the previous one, and update with your path
#driver_path = Service('/Users/claro/Documents/GitHub/PythonCamp2026/Day02/Day02_Part02/Lecture/chromedriver')
# if on Windows, may need to add '.exe' at the end of path
#driver = webdriver.Chrome(service = driver_path)



2. Start the web driver

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome()

driver.get("https://polisci.washu.edu/people")

time.sleep(2)

# find all links to individual people
people = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/people/"]')

# click the first visible profile
for person in people:
    if person.is_displayed() and person.text.strip():
        print("Opening:", person.text)
        person.click()
        break


##### Scraping Tips
- Google Chrome is better to track nodes and page sources
- Inspect the source and get to know your document/website!
- Selenium—Use the ’Copy Xpath’ command if you’re having troubles (Find it in "Inspect" in Google Chrome)
- Use time breaks to avoid being blocked and be polite
- Check the Terms of Service (whether you obey them or not). Please don't get sued. 


##### More on Selenium: https://selenium-python.readthedocs.io/locating-elements.html

### 2. Reading and Writing Files 

#### 2.1 Reading Files

1. Import libraries

In [ ]:
import os

2. View your working directory

In [ ]:
os.getcwd()
import sys


3. Set your working directory 

In [ ]:
# os.chdir('/Users/claro/Documents/GitHub/PythonCamp2026/Day02/Day02_Part02/Lecture')

4. Read lines from the file

In [ ]:
# Read all lines as one string
with open('readfile.txt') as f:
  the_whole_thing = f.read()
  print(the_whole_thing)

In [ ]:
# Read line by line
with open('readfile.txt') as f:
  lines_list = f.readlines()
  for l in lines_list:
    print(l)

In [ ]:
# More efficiently, we can loop over the file object (i.e. we don't need the variable lines)
with open('readfile.txt') as f:   
  for l in f:
    print(l)

In [ ]:
# We can also manually open and close files
# I never do this
f =  open('readfile.txt')
print(f.read())
f.close()

Tips: 
- Try to minimize the number of times you open and close flies
- It is very expensive and consumes limited resources --> if too many, it leads to errors 

_Source: https://www.geeksforgeeks.org/context-manager-in-python/_


In [ ]:
# file_descriptors = [] 
#for x in range(100000000000): 
#    file_descriptors.append(open('readfile.txt')) 

In [ ]:
open('readfile.txt')

#### 2.2 Writing Files

1. Writing files is easy, but be careful not to overwrite the content you actually want
2. See https://stackabuse.com/file-handling-in-python/ for more options

* We need to use the option 'w'

In [ ]:
with open('test_writefile.txt', 'w') as f:
  ## wipes the file clean and opens it
  f.write("Hi guys.")
  f.write("Does this go on the second line?")
  f.writelines(['a\n', 'b\n', 'c\n'])

In [ ]:
# We use 'a' to append new information to it
with open('test_writefile.txt', 'a') as f:
  f.write("I got appended!")

##### Writing CSV files (pre-pandas)

1. Import csv

In [ ]:
import csv

2. Open a file stream and create a `csv` writer object

In [ ]:
# Open a file stream and create a CSV writer object
with open('test_writecsv.csv', 'w') as f:
  my_writer = csv.writer(f)
  for i in range(1, 100):
    my_writer.writerow([i, i-1])

3. Now read the `csv` file

In [ ]:
with open('test_writecsv.csv', 'r') as f:
  my_reader = csv.reader(f)
  mydat = []
  for row in my_reader:
    mydat.append(row)
print(mydat[0],"\n", mydat[1],"\n", mydat[2],"\n", mydat[3])

4. Add column names 

In [ ]:
# Note that we are writing a new file
with open('test_csvfields.csv', 'w') as f:
  my_writer = csv.DictWriter(f, fieldnames = ("A", "B"))
  my_writer.writeheader()
  for i in range(1, 100):
    my_writer.writerow({"B":i, "A":i-1})

5. Read the new file

In [ ]:
b = 0
with open('test_csvfields.csv', 'r') as f:
  my_reader = csv.DictReader(f)
  for row in my_reader:
      if b<5:
          print(row)
          b +=1

##### Some Tips

- Tip 1: We may find useful to save webpages for collecting data (to `.html` files)

In [ ]:
import os

Then, we can parse a page that is already saved on your computer even without access to internet. 

In [ ]:
with open('polisci_ppl.html') as f:
  myfile = f.read()
  soup = BeautifulSoup(myfile)
# soup.prettify()

- Tip 2: You may also write directly from a website to a `csv` file. This is good practice as it ensures a break 10 hours into the process does not erase all of your data. 
- Tip 3: Use Exception Handling techniques that we covered in Day03

In [ ]:
# Copyright of the original version:

# Copyright (c) 2014 Matt Dickenson
# 
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
# 
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
# 
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.